# Notebook 04 Data Wrangling & Preprocessing: Study Habit
**Proyek Analisis Data: HAPI (Human Activity Pattern Intelligence)**

### Profil Proyek & Tim
- **Tema Capstone:** Healthy Lives & Well-being
- **Target User:** Mahasiswa (fokus pada aspek akademik & lingkungan belajar)
- **Tujuan Proyek:** Pengembangan *platform* Webapp untuk *mood management & tracker* serta diagnosis mandiri tingkat kelelahan (*fatigue*).
- **Tim DS:** Greycia Febrina Michelle (CDCC700D6X2644) & Khazel Hayfa Yosmi (CDCC308D6X0629)

### Tujuan Notebook
Melakukan proses Data Wrangling secara *end-to-end* (Gathering → Assessing → Cleaning) dan Preprocessing pada dataset Study Habits and Daily Lifestyles of Students. Langkah ini bertujuan untuk menghasilkan data fitur perilaku belajar yang bersih, terstruktur, dan siap dikirim ke AI Engineer.

### Peran Dataset
Dataset yang digunakan dalam tahapan ini bersumber dari Kaggle: [Study Habits and Activities of Students](https://www.kaggle.com/datasets/afnansaifafnan/study-habits-and-activities-of-students).

Dataset ini menyediakan fitur perilaku harian mahasiswa yang mencakup jam belajar, pola tidur, aktivitas fisik, hingga interaksi sosial. 

Data yang telah diproses nantinya akan digunakan sebagai input untuk **Fatigue Score Fusion Model** bersama dengan hasil output dari instrumen MBI dan pemrosesan teks NLP pada aplikasi HAPI.

### Kolom Target
- `stress_level`: nilai numerik yang berfungsi sebagai target regresi untuk model perilaku.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / 'data').exists():
        break
    ROOT = ROOT.parent

RAW_PATH   = ROOT / 'data' / 'raw' / 'model_ready' / 'student_lifestyle_dirty.csv'
CLEAN_PATH = ROOT / 'data' / 'clean' / 'model_ready' / 'student_lifestyle_clean.csv'
PREP_PATH  = ROOT / 'data' / 'preprocessed' / 'student_lifestyle_preprocessed.csv'

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
PREP_PATH.parent.mkdir(parents=True, exist_ok=True)

# Nama kolom sesuai dataset aktual (Title_Case)
FEATURE_COLS = [
    'Study_Hours_Per_Day',
    'Sleep_Hours_Per_Day',
    'Physical_Activity_Hours_Per_Day',
    'Extracurricular_Hours_Per_Day',
    'Social_Hours_Per_Day',
]
DROP_COLS  = ['GPA', 'Student_ID']
TARGET_COL = 'Stress_Level'

print(f'ROOT     : {ROOT}')
print(f'File ada : {RAW_PATH.exists()}')
print('Setup selesai.')

ROOT     : d:\Proyek_Analisis_Burnout
File ada : True
Setup selesai.


**Dokumentasi Setup Data: Student Lifestyle**  
**Proyek HAPI (Human Activity Pattern Intelligence)**

Kami menyusun script ini sebagai langkah awal pemrosesan data gaya hidup mahasiswa. Karena dataset ini akan menjadi masukan penting bagi model kami, kami perlu menata ulang strukturnya agar siap digunakan oleh rekan AI Engineer di tahap selanjutnya.

**Operasi Utama**

**Path Mapping**  
Kami memakai `pathlib` supaya script kami otomatis mendeteksi lokasi direktori proyek. Jadi, tidak ada lagi isu path yang rusak atau tidak terbaca saat kami menjalankan kode di laptop yang berbeda-beda.

**Directory Setup**  
Kami membuat mekanisme otomatis untuk memastikan folder data tersedia. Kami membagi alur penyimpanan menjadi tiga tahap agar proses pembersihan data tidak menimpa data aslinya:

- `RAW_PATH`: Titik awal file `student_lifestyle_dirty.csv`.
- `CLEAN_PATH` & `PREP_PATH`: Folder tujuan untuk menyimpan file yang sudah dibersihkan dan hasil rekayasa fitur.

**Schema Definition**  
Kami melakukan seleksi kolom yang sangat spesifik agar model nantinya tidak terbebani oleh data yang tidak relevan:

- `FEATURE_COLS`: Fokus pada durasi aktivitas harian seperti belajar, tidur, aktivitas fisik, kegiatan ekstrakurikuler, dan interaksi sosial.
- `DROP_COLS`: Kami membuang `GPA` dan `Student_ID` karena tidak berpengaruh langsung pada model prediksi kelelahan yang sedang kami bangun.
- `TARGET_COL`: Target regresi kita adalah `Stress_Level`.

**Alur Kerja Data**

```text
student_lifestyle_dirty.csv
          │
          ▼
      RAW_PATH
          │
          ▼
  Data Cleaning
          │
          ▼
     CLEAN_PATH
          │
          ▼
 Feature Engineering
          │
          ▼
     PREP_PATH
          │
          ▼
  Fatigue Fusion Model
```

**Catatan Teknis**

Kami memformat konfigurasi `pandas` agar menampilkan kolom secara utuh dan membatasi presisi angka desimal sebanyak **4 digit**. Langkah ini sengaja kami ambil supaya data yang kami kirimkan ke tim AI Engineer memiliki format yang konsisten dan akurat untuk tahap Fusion Model. Script sudah berhasil mendeteksi file di `d:\Proyek_Analisis_Burnout` dan siap untuk diproses lebih lanjut.

## Gathering Data

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Dataset dimuat: {df_raw.shape[0]:,} baris, {df_raw.shape[1]} kolom')
print(f'Kolom: {df_raw.columns.tolist()}')
df_raw.head()

Dataset dimuat: 2,335 baris, 9 kolom
Kolom: ['Student_ID', 'Study_Hours_Per_Day', 'Extracurricular_Hours_Per_Day', 'Sleep_Hours_Per_Day', 'Social_Hours_Per_Day', 'Physical_Activity_Hours_Per_Day', 'GPA', 'Stress_Level', 'row_status']


,Student_ID,Study_Hours_Per_Day,Extracurricular_Hours_Per_Day,Sleep_Hours_Per_Day,Social_Hours_Per_Day,Physical_Activity_Hours_Per_Day,GPA,Stress_Level,row_status
0,1442,7.6000,3.2000,8.7000,3.0000,1.5000,2.8600,Moderate,clean
1,31,6.2000,2.9000,6.8000,3.8000,4.3000,3.0500,Moderate,clean
2,1533,9.9000,1.1000,5.2000,4.2000,3.6000,3.3000,High,clean
3,259,9.6000,0.8000,8.1000,3.5000,2.0000,3.3100,High,clean
4,1567,6.0000,1.8000,9.0000,3.5000,3.7000,2.9200,Moderate,clean


**Pemuatan Dataset Keseimbangan Hidup Mahasiswa**

**Penjelasan Singkat**

- `pd.read_csv(RAW_PATH)`: Membaca file data mentah dari direktori yang ditentukan.

- `df_raw.shape`: Menampilkan dimensi data, yaitu total baris dan jumlah fitur yang tersedia.

- `df_raw.columns.tolist()`: Menampilkan daftar kolom untuk memastikan seluruh atribut sudah terbaca sesuai kebutuhan.

- `df_raw.head()`: Menampilkan lima baris pertama agar kami bisa melakukan verifikasi awal terhadap konten data.

**Ringkasan Data**

Hasil pemuatan menunjukkan dataset terdiri dari **2.335 baris** dan **9 kolom**. Struktur kolom mencakup identitas unik mahasiswa, rincian durasi aktivitas harian (studi, ekstrakurikuler, tidur, sosial, dan fisik), serta metrik performa akademik (GPA) dan tingkat stres. Berdasarkan cuplikan tersebut, data terlihat sudah dalam format yang terstruktur dengan rapi dan siap kami bawa ke tahap pembersihan atau analisis korelasi antar variabel.

## Assessing Data

In [3]:
print('Shape')
print(f'Baris: {df_raw.shape[0]:,} | Kolom: {df_raw.shape[1]}')

Shape
Baris: 2,335 | Kolom: 9


In [4]:
print('Tipe Data')
print(df_raw.dtypes)

Tipe Data
Student_ID                           int64
Study_Hours_Per_Day                float64
Extracurricular_Hours_Per_Day      float64
Sleep_Hours_Per_Day                float64
Social_Hours_Per_Day               float64
Physical_Activity_Hours_Per_Day    float64
GPA                                float64
Stress_Level                        object
row_status                          object
dtype: object


In [5]:
print('Missing Values')
missing = df_raw.isnull().sum()
mv_df = pd.DataFrame({'jumlah': missing, 'persen': (missing/len(df_raw)*100).round(2)})
print(mv_df[mv_df['jumlah'] > 0] if mv_df['jumlah'].sum() > 0 else 'Tidak ada missing values.')

Missing Values
                     jumlah  persen
Study_Hours_Per_Day      60  2.5700
Sleep_Hours_Per_Day      50  2.1400
Stress_Level             40  1.7100


In [6]:
print('Duplikat')
print(f'Baris duplikat: {df_raw.duplicated().sum():,}')

Duplikat
Baris duplikat: 0


In [7]:
print('Statistik Deskriptif')
print(df_raw.describe().round(3))

Statistik Deskriptif
       Student_ID  Study_Hours_Per_Day  Extracurricular_Hours_Per_Day  \
count   2335.0000            2275.0000                      2335.0000   
mean   13961.2400               7.6380                         1.9960   
std    31697.6220               2.0740                         1.1500   
min        1.0000               5.0000                         0.0000   
25%      584.5000               6.3000                         1.0000   
50%     1168.0000               7.4000                         2.0000   
75%     1751.5000               8.7300                         3.0000   
max    99039.0000              24.8700                         4.0000   

       Sleep_Hours_Per_Day  Social_Hours_Per_Day  \
count            2285.0000             2335.0000   
mean                7.4180                2.7020   
std                 1.7100                1.6690   
min                -4.7300                0.0000   
25%                 6.2000                1.3000   
50%      

In [8]:
print('Distribusi Row Status (Audit Masalah yang Disuntikkan)')
print(df_raw['row_status'].value_counts())

Distribusi Row Status (Audit Masalah yang Disuntikkan)
row_status
clean                                 2000
dirty_typo_Stress_Level                100
dirty_missing_Study_Hours               60
dirty_missing_Sleep_Hours               50
dirty_missing_Stress_Level              40
dirty_duplicate                         40
dirty_outlier_Study_Hours_high          25
dirty_outlier_Sleep_Hours_negative      20
Name: count, dtype: int64


In [9]:
print('Distribusi Target')
print(f'Tipe data : {df_raw[TARGET_COL].dtype}')
print(f'Nilai unik: {sorted(df_raw[TARGET_COL].dropna().unique())}')
print('\nDistribusi:')
print(df_raw[TARGET_COL].value_counts())
print('\nProporsi:')
print(df_raw[TARGET_COL].value_counts(normalize=True).round(3))

Distribusi Target
Tipe data : object
Nilai unik: ['HIGH', 'Hi', 'High', 'LOW', 'Lo', 'Low', 'MODERATE', 'Med', 'Moderate', 'high', 'low', 'moderate']

Distribusi:
Stress_Level
High        1125
Moderate     741
Low          329
HIGH          20
Hi            20
high          11
moderate      11
Med           10
Lo             8
LOW            8
MODERATE       6
low            6
Name: count, dtype: int64

Proporsi:
Stress_Level
High       0.4900
Moderate   0.3230
Low        0.1430
HIGH       0.0090
Hi         0.0090
high       0.0050
moderate   0.0050
Med        0.0040
Lo         0.0030
LOW        0.0030
MODERATE   0.0030
low        0.0030
Name: proportion, dtype: float64


**Profil Dataset**

Kami mengolah **2.335 entri** dengan **9 kolom** yang mencakup durasi aktivitas harian mahasiswa, performa akademik, dan tingkat stres yang dirasakan. Secara struktural, sebagian besar tipe data sudah terbaca dengan benar, namun kami menemukan beberapa masalah kualitas pada label kategorikal dan nilai numerik yang perlu segera diseragamkan.

**Temuan Kritis**

Kami mengidentifikasi beberapa titik masalah yang akan mengganggu hasil analisis jika dibiarkan:

- **Data Kosong:** Terdapat celah data pada variabel `Study_Hours_Per_Day`, `Sleep_Hours_Per_Day`, dan `Stress_Level` yang mencakup sekitar **1,7% hingga 2,5%** dari total dataset.

- **Inkonsistensi Kategori:** Label `Stress_Level` memiliki banyak variasi penulisan yang tidak standar, seperti `Hi`, `high`, `Lo`, `low`, `Med`, dan `MODERATE`. Hal ini akan mengacaukan klasifikasi jika tidak segera dipetakan ulang ke kategori baku (`Low`, `Moderate`, `High`).

- **Anomali Logika:** Kami menemukan data jam tidur bernilai negatif dan durasi jam belajar yang tidak realistis (di atas 24 jam), yang menunjukkan adanya kesalahan pada input data atau sistem pelaporan.

**Ringkasan Hasil Assessing**

| Masalah | Detail | Tindakan |
|----------|----------|----------|
| Missing values | Study_Hours, Sleep_Hours, Stress_Level | Hapus baris dengan nilai kosong untuk menjaga integritas model |
| Outlier/invalid | Jam tidur negatif, jam belajar tidak logis | Hapus baris yang mengandung anomali tersebut |
| Typo Stress_Level | 'Hi', 'high', 'Lo', 'low', 'Med', 'Moderate', dsb. | Koreksi dan seragamkan ke kategori standar (Low, Moderate, High) |
| Duplikat | 40 baris duplikat terdeteksi | Hapus baris duplikat |
| Kolom tidak diperlukan | row_status | Drop kolom sebelum tahap pelatihan model |

## Cleaning Data

In [10]:
df = df_raw.copy()
before = len(df)

df = df[df['row_status'] == 'clean'].copy()
print(f'Drop Baris Kotor: {before - len(df):,} dihapus → sisa {len(df):,}')

Drop Baris Kotor: 335 dihapus → sisa 2,000


In [11]:
df = df.drop(columns=['row_status'])
print('Drop Kolom Row Status ✓')
print(f'Missing Values Tersisa: {df.isnull().sum().sum()}')
print(f'Kolom: {df.columns.tolist()}')

Drop Kolom Row Status ✓
Missing Values Tersisa: 0
Kolom: ['Student_ID', 'Study_Hours_Per_Day', 'Extracurricular_Hours_Per_Day', 'Sleep_Hours_Per_Day', 'Social_Hours_Per_Day', 'Physical_Activity_Hours_Per_Day', 'GPA', 'Stress_Level']


In [12]:
cols_to_drop = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop)

print(f'Drop Kolom: {cols_to_drop}')
print(f'Kolom Tersisa: {df.columns.tolist()}')

Drop Kolom: ['GPA', 'Student_ID']
Kolom Tersisa: ['Study_Hours_Per_Day', 'Extracurricular_Hours_Per_Day', 'Sleep_Hours_Per_Day', 'Social_Hours_Per_Day', 'Physical_Activity_Hours_Per_Day', 'Stress_Level']


In [13]:
hour_constraints = {
    'Study_Hours_Per_Day': (0, 16),
    'Sleep_Hours_Per_Day': (0, 14),
    'Physical_Activity_Hours_Per_Day': (0, 8),
    'Extracurricular_Hours_Per_Day': (0, 8),
    'Social_Hours_Per_Day': (0, 10),
}

for col, (lo, hi) in hour_constraints.items():
    if col in df.columns:
        n = ((df[col] < lo) | (df[col] > hi)).sum()
        df[col] = df[col].clip(lo, hi)
        if n > 0:
            print(f'Clip {col} ke [{lo},{hi}]: {n} nilai')

valid_stress = {'Low', 'Moderate', 'High'}
invalid = df[~df[TARGET_COL].isin(valid_stress)]

assert len(invalid) == 0, f'Label tidak valid: {df[TARGET_COL].unique()}'
print(f'Stress Level Valid: {sorted(df[TARGET_COL].unique())} ✓')

assert df.isnull().sum().sum() == 0, 'Masih ada missing values!'
print('Missing Values: 0 ✓')

print(f'\nShape Akhir Cleaned: {df.shape}')
df.head(3)

Clip Physical_Activity_Hours_Per_Day ke [0,8]: 174 nilai
Stress Level Valid: ['High', 'Low', 'Moderate'] ✓
Missing Values: 0 ✓

Shape Akhir Cleaned: (2000, 6)


,Study_Hours_Per_Day,Extracurricular_Hours_Per_Day,Sleep_Hours_Per_Day,Social_Hours_Per_Day,Physical_Activity_Hours_Per_Day,Stress_Level
0,7.6000,3.2000,8.7000,3.0000,1.5000,Moderate
1,6.2000,2.9000,6.8000,3.8000,4.3000,Moderate
2,9.9000,1.1000,5.2000,4.2000,3.6000,High


In [14]:
df.to_csv(CLEAN_PATH, index=False)
print(f'Dataset Cleaned Disimpan Ke: {CLEAN_PATH}')

Dataset Cleaned Disimpan Ke: d:\Proyek_Analisis_Burnout\data\clean\model_ready\student_lifestyle_clean.csv


**Pembersihan Data untuk Dataset Gaya Hidup Mahasiswa**

Kami menyelesaikan pembersihan data untuk dataset gaya hidup mahasiswa agar siap digunakan ke tahap analisis selanjutnya. Fokus kami adalah memastikan setiap input jam aktivitas berada dalam rentang yang masuk akal dan membuang kolom yang tidak diperlukan.

**Langkah-Langkah Teknis**

- **Penyaringan Data Mentah:** Kami membuang **335 baris** yang ditandai sebagai kotor agar dataset hanya berisi data berkualitas untuk pelatihan.

- **Pemangkasan Kolom:** Kami menghapus kolom `row_status`, `Student_ID`, dan `GPA` karena sudah tidak relevan dengan model yang akan kami bangun.

- **Normalisasi Rentang Nilai:** Kami menerapkan clipping pada kolom waktu aktivitas seperti belajar, tidur, olahraga, ekstrakurikuler, dan sosial ke dalam rentang waktu harian yang logis. Sebagai contoh, ada **174 nilai** pada kolom olahraga yang kami sesuaikan agar tidak melebihi batas wajar.

- **Validasi Integritas:** Kami melakukan pengecekan untuk memastikan kolom `Stress_Level` hanya berisi label yang valid yaitu `Low`, `Moderate`, dan `High`. Selain itu, kami memastikan tidak ada nilai kosong yang tersisa.

- **Verifikasi Akhir:** Kami memastikan dataset sudah benar-benar bersih dari missing values sebelum menyimpannya ke tahap berikutnya.

**Hasil Akhir**

Setelah proses filter dan normalisasi tadi, kami mendapatkan dataset bersih dengan total **2.000 baris** dan **6 kolom** siap pakai. File final ini sudah kami simpan di `student_lifestyle_clean.csv` dan siap untuk masuk ke tahap pemodelan.

## Preprocessing & Feature Engineering

In [15]:
df_prep = pd.read_csv(CLEAN_PATH)

hour_cols = [c for c in FEATURE_COLS if c in df_prep.columns]
df_prep['total_active_hours'] = df_prep[hour_cols].sum(axis=1).round(4)

print(f'total_active_hours — mean: {df_prep["total_active_hours"].mean():.2f}')

total_active_hours — mean: 23.89


### Feature Engineering

In [16]:
if 'Study_Hours_Per_Day' in df_prep.columns:
    df_prep['study_ratio'] = (
        df_prep['Study_Hours_Per_Day'] /
        df_prep['total_active_hours'].replace(0, np.nan)
    ).fillna(0).clip(0, 1).round(4)

    print(f'study_ratio — mean: {df_prep["study_ratio"].mean():.4f}')

study_ratio — mean: 0.3127


In [17]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scale_cols = [c for c in FEATURE_COLS if c in df_prep.columns]

scaled_data = scaler.fit_transform(df_prep[scale_cols])
scaled_df = pd.DataFrame(scaled_data, columns=[f'{c}_scaled' for c in scale_cols])

df_prep = pd.concat([df_prep, scaled_df], axis=1)

print(f'Kolom Scaled Dibuat: {scaled_df.columns.tolist()}')

Kolom Scaled Dibuat: ['Study_Hours_Per_Day_scaled', 'Sleep_Hours_Per_Day_scaled', 'Physical_Activity_Hours_Per_Day_scaled', 'Extracurricular_Hours_Per_Day_scaled', 'Social_Hours_Per_Day_scaled']


### Encoding dan Finalisasi

In [18]:
stress_map = {'Low': 0, 'Moderate': 1, 'High': 2}
df_prep['Stress_Level_encoded'] = df_prep[TARGET_COL].map(stress_map)

n_null = df_prep['Stress_Level_encoded'].isnull().sum()
assert n_null == 0, f'{n_null} nilai gagal di-encode — cek nilai unik kolom target'

print(f'ENC Stress Level: {stress_map}')
print(df_prep[[TARGET_COL, 'Stress_Level_encoded']].value_counts().sort_index())

all_feature_cols = [
    c for c in df_prep.columns
    if c not in [TARGET_COL, 'Stress_Level_encoded']
]

leakage = set([TARGET_COL, 'Stress_Level_encoded']) & set(all_feature_cols)
assert len(leakage) == 0, f'LEAKAGE: {leakage}'

print('Leakage Check: BERSIH ✓')

ENC Stress Level: {'Low': 0, 'Moderate': 1, 'High': 2}
Stress_Level  Stress_Level_encoded
High          2                       1029
Low           0                        297
Moderate      1                        674
Name: count, dtype: int64
Leakage Check: BERSIH ✓


In [19]:
df_prep.to_csv(PREP_PATH, index=False)

print(f'Dataset Model-Ready Disimpan Ke: {PREP_PATH}')

print('\nRingkasan Final')
print(f'Total Baris: {len(df_prep):,}')
print(f'Total Kolom: {df_prep.shape[1]}')
print(f'Missing Values: {df_prep.isnull().sum().sum()}')
print(f'Target (Raw): {TARGET_COL} → Low / Moderate / High')
print('Target (Encoded): Stress_Level_encoded → 0 / 1 / 2')
print('Leakage: BERSIH')

print('\nDistribusi Target')
print(df_prep[TARGET_COL].value_counts())

Dataset Model-Ready Disimpan Ke: d:\Proyek_Analisis_Burnout\data\preprocessed\student_lifestyle_preprocessed.csv

Ringkasan Final
Total Baris: 2,000
Total Kolom: 14
Missing Values: 0
Target (Raw): Stress_Level → Low / Moderate / High
Target (Encoded): Stress_Level_encoded → 0 / 1 / 2
Leakage: BERSIH

Distribusi Target
Stress_Level
High        1029
Moderate     674
Low          297
Name: count, dtype: int64


**Rekayasa Fitur untuk Analisis Gaya Hidup Mahasiswa**

Kami telah merampungkan persiapan data agar siap digunakan ke dalam model machine learning. Fokus utama kami adalah menciptakan representasi data yang lebih informatif melalui perhitungan rasio aktivitas serta standarisasi skala fitur.

**Verifikasi dan Transformasi Data**

- **Fitur Turunan:** Kami menghitung `total_active_hours` untuk menjumlahkan semua durasi kegiatan harian, kemudian menggunakan nilai tersebut untuk membuat `study_ratio` guna memahami proporsi waktu belajar mahasiswa dibanding aktivitas lainnya.

- **Penskalaan Fitur:** Kami menerapkan `MinMaxScaler` untuk menormalisasi kolom durasi kegiatan (belajar, tidur, olahraga, sosial, dan ekstrakurikuler) ke rentang **0 hingga 1**. Langkah ini penting agar model tidak bias karena perbedaan skala satuan antar fitur.

- **Encoding Target:** Label `Stress_Level` (`Low`, `Moderate`, `High`) kami konversi secara manual ke format numerik (`0`, `1`, `2`) untuk memudahkan proses klasifikasi oleh algoritma.

**Jaminan Kualitas**

Kami menerapkan beberapa prosedur untuk mencegah kegagalan model di masa depan:

- **Cek Kebocoran Data:** Kami memastikan variabel target `Stress_Level` tidak tercampur ke dalam fitur input. Langkah ini krusial agar model tidak melakukan prediksi berdasarkan informasi yang seharusnya belum tersedia.

- **Struktur Final:** Dataset kini memiliki **14 kolom** yang mencakup data dasar, fitur hasil penskalaan, fitur turunan, serta target yang sudah di-encode.

- **Penyimpanan:** File hasil akhir sudah tersimpan di `student_lifestyle_preprocessed.csv`.

**Ringkasan**

Ringkasnya, kami sudah memiliki **2.000 baris** data yang bersih dan sudah terstandarisasi. Dataset ini siap untuk diuji ke berbagai model klasifikasi. Langkah berikutnya adalah mulai melatih model untuk memprediksi tingkat stres mahasiswa berdasarkan profil gaya hidup mereka.